# Del corrimiento de los pozos al campo de los imanes

Con imanes, el pozo 85F3 de los dos canales del fotodiodo se separa: medimos $\Delta = (-122 \pm 12)$ MHz con los imanes en un sentido y $\Delta = (+112 \pm 10)$ MHz con los imanes invertidos (sección 4.2 del informe).

Este notebook muestra cómo pasamos de esa separación, en MHz, al campo $B$, en gauss. Usamos solo datos de D. A. Steck, *Rubidium 85 D Line Data* ([steck.us/alkalidata](https://steck.us/alkalidata/)), y un promedio pesado. Las tablas y figuras que citamos son las de ese documento.



## 1. Los estados del átomo

El pozo 85F3 lo producen átomos de ⁸⁵Rb cuyo electrón externo salta del estado fundamental $5S_{1/2}$ al excitado $5P_{1/2}$ (la línea D1, 795 nm).

Cada uno de esos estados se divide en dos niveles. La energía depende de cómo se combinan el momento angular del electrón ($J = 1/2$) y el espín del núcleo ($I = 5/2$, Tabla 2). El momento angular total del átomo se llama $F$, y puede valer $I - J = 2$ o $I + J = 3$. En el estado excitado lo escribimos $F'$.

Cada nivel tiene además $2F + 1$ subniveles. Los distingue $m_F$, la proyección de $F$ sobre la dirección del campo, que va de $-F$ a $F$ de a uno. Así, los estados son

$$|5S_{1/2}, F, m_F\rangle \qquad \text{y} \qquad |5P_{1/2}, F', m_{F'}\rangle,$$

con 7 + 5 = 12 estados fundamentales y otros 12 excitados.

**Sin campo**, la energía depende del estado ($S$ o $P$) y de $F$, pero no de $m_F$: hay solo cuatro energías, las de la Figura 3.

In [1]:
from fractions import Fraction as Fr

def subniveles(F):
    return list(range(-F, F + 1))

for nombre, F in [("5S1/2, F=3", 3), ("5S1/2, F=2", 2), ("5P1/2, F'=3", 3), ("5P1/2, F'=2", 2)]:
    print(f"{nombre}: m_F = {subniveles(F)}  ({2*F + 1} estados)")

5S1/2, F=3: m_F = [-3, -2, -1, 0, 1, 2, 3]  (7 estados)
5S1/2, F=2: m_F = [-2, -1, 0, 1, 2]  (5 estados)
5P1/2, F'=3: m_F = [-3, -2, -1, 0, 1, 2, 3]  (7 estados)
5P1/2, F'=2: m_F = [-2, -1, 0, 1, 2]  (5 estados)


## 2. Qué saltos forman el pozo 85F3

El pozo 85F3 lo forman los átomos que **parten de $F = 3$**. Desde ahí pueden llegar a $F' = 2$ o a $F' = 3$. Esos dos saltos están separados por 361,58 MHz (Figura 3), menos que el ancho Doppler de cada uno, así que forman un solo pozo. Los átomos que parten de $F = 2$ forman otro pozo, 3 GHz más arriba, y no entran en este cálculo.

**Cuántos átomos hay en cada estado.** La diferencia de energía entre los estados fundamentales (3 GHz entre $F = 2$ y $F = 3$, mucho menos entre subniveles) es mucho menor que $k_B T$ a temperatura ambiente (unos 6 THz). Por eso los átomos se reparten por igual entre los 12 estados fundamentales, y cada $|5S_{1/2}, 3, m_F\rangle$ tiene la misma población.

## 3. Qué hace el campo con cada estado

Con campo, cada estado se corre una energía que depende de su $m_F$ (Ec. 24):

$$\Delta E = \mu_B\, g_F\, m_F\, B.$$

Dividiendo por $h$ obtenemos el corrimiento en frecuencia,

$$\Delta\nu = \frac{\mu_B}{h}\, g_F\, m_F\, B, \qquad \frac{\mu_B}{h} = 1{,}3996\ \text{MHz/G}\ \text{(Tabla 1)}.$$

El factor $g_F$ es uno por nivel, y Steck lo escribe junto a cada nivel en la Figura 3:

| Nivel | $g_F$ |
|---|---|
| $5S_{1/2}$, $F = 3$ | $+1/3$ |
| $5P_{1/2}$, $F' = 2$ | $-1/9$ |
| $5P_{1/2}$, $F' = 3$ | $+1/9$ |

Sin campo, los 7 subniveles de $F = 3$ tienen la misma energía; con campo, se abren en 7 energías equiespaciadas.

In [2]:
MU_B = 1.3996                  # mu_B / h, en MHz/G (Steck, Tabla 1)
g = {"S3": Fr(1, 3),           # 5S1/2, F = 3   (Steck, Figura 3)
     "P2": Fr(-1, 9),          # 5P1/2, F' = 2
     "P3": Fr(1, 9)}           # 5P1/2, F' = 3

def corrimiento_estado(nivel, m):
    # Corrimiento de un estado, en MHz por gauss de campo
    return MU_B * float(g[nivel]) * m

print("Corrimiento de cada estado de F = 3 (MHz/G):")
for m in subniveles(3):
    print(f"  |5S1/2, 3, {m:+d}>: {corrimiento_estado('S3', m):+.3f}")

Corrimiento de cada estado de F = 3 (MHz/G):
  |5S1/2, 3, -3>: -1.400
  |5S1/2, 3, -2>: -0.933
  |5S1/2, 3, -1>: -0.467
  |5S1/2, 3, +0>: +0.000
  |5S1/2, 3, +1>: +0.467
  |5S1/2, 3, +2>: +0.933
  |5S1/2, 3, +3>: +1.400


## 4. Qué saltos produce la luz

La luz polarizada circularmente lleva una unidad de momento angular, y el átomo la absorbe al saltar. Por eso:

- la luz $\sigma^+$ solo produce saltos con $m_{F'} = m_F + 1$;
- la luz $\sigma^-$ solo produce saltos con $m_{F'} = m_F - 1$.

Steck lo escribe en el título de las Tablas 15 ($\sigma^+$) y 17 ($\sigma^-$). Cada canal del fotodiodo ve una de las dos polarizaciones.

Un salto existe solo si el estado de llegada existe: $|m_{F'}| \le F'$. Por ejemplo, desde $m_F = +3$ no hay salto $\sigma^+$, porque $m_{F'} = +4$ no existe en ningún nivel.

In [3]:
def saltos(q):
    # Saltos desde F = 3 con luz sigma^q (q = +1 o -1): lista de (F', m_F, m_F')
    lista = []
    for Fp in (2, 3):
        for m in subniveles(3):
            mp = m + q
            if abs(mp) <= Fp:
                lista.append((Fp, m, mp))
    return lista

for q, nombre in [(+1, "sigma+"), (-1, "sigma-")]:
    print(f"{nombre}: {len(saltos(q))} saltos")
for Fp, m, mp in saltos(+1):
    print(f"  |5S1/2, 3, {m:+d}>  ->  |5P1/2, {Fp}, {mp:+d}>")

sigma+: 11 saltos
sigma-: 11 saltos
  |5S1/2, 3, -3>  ->  |5P1/2, 2, -2>
  |5S1/2, 3, -2>  ->  |5P1/2, 2, -1>
  |5S1/2, 3, -1>  ->  |5P1/2, 2, +0>
  |5S1/2, 3, +0>  ->  |5P1/2, 2, +1>
  |5S1/2, 3, +1>  ->  |5P1/2, 2, +2>
  |5S1/2, 3, -3>  ->  |5P1/2, 3, -2>
  |5S1/2, 3, -2>  ->  |5P1/2, 3, -1>
  |5S1/2, 3, -1>  ->  |5P1/2, 3, +0>
  |5S1/2, 3, +0>  ->  |5P1/2, 3, +1>
  |5S1/2, 3, +1>  ->  |5P1/2, 3, +2>
  |5S1/2, 3, +2>  ->  |5P1/2, 3, +3>


## 5. Cuánto se corre cada salto

La frecuencia de un salto es la diferencia de energía entre el estado de llegada y el de partida, dividida por $h$. Con campo, los dos estados se corren, así que la frecuencia del salto se corre, respecto de la del salto sin campo,

$$\Delta\nu_\text{salto} = \Delta\nu_\text{llegada} - \Delta\nu_\text{partida} = \frac{\mu_B}{h}\,\big(g_{F'}\, m_{F'} - g_F\, m_F\big)\, B.$$

**Ejemplo.** En el salto $|5S_{1/2}, 3, -3\rangle \to |5P_{1/2}, 2, -2\rangle$:

- el estado de partida se corre $1{,}3996 \cdot (1/3) \cdot (-3) = -1{,}400$ MHz/G: baja;
- el de llegada se corre $1{,}3996 \cdot (-1/9) \cdot (-2) = +0{,}311$ MHz/G: sube;
- la diferencia de energía crece $0{,}311 - (-1{,}400) = 1{,}711$ MHz/G, y el salto se corre esa cantidad hacia frecuencias mayores.

Con 79 G, este salto se corre unos 135 MHz.

In [4]:
def corrimiento_salto(Fp, m, mp):
    # Corrimiento de la frecuencia de un salto, en MHz por gauss
    return corrimiento_estado(f"P{Fp}", mp) - corrimiento_estado("S3", m)

print(f"Ejemplo: {corrimiento_salto(2, -3, -2):+.3f} MHz/G")

Ejemplo: +1.711 MHz/G


## 6. Cuánto absorbe cada salto

No todos los saltos absorben lo mismo. La probabilidad de cada salto depende de un coeficiente que da Steck: la Tabla 15 para $\sigma^+$ y la Tabla 17 para $\sigma^-$, en la fila de cada $F'$ y la columna de cada $m_F$. Steck da cada coeficiente como una raíz cuadrada (por ejemplo, $\sqrt{5/9}$).

La luz que absorbe cada salto es proporcional al **cuadrado** de su coeficiente. Como todos los estados de partida tienen la misma población, ese cuadrado alcanza para comparar los saltos: lo llamamos **peso**.

Abajo copiamos los cuadrados de las Tablas 15 y 17 (el signo de los coeficientes no importa).

In [5]:
# Cuadrados de los coeficientes de Steck, por (F', m_F)
peso = {
    +1: {  # Tabla 15, sigma+ (m_F' = m_F + 1)
        (2, -3): Fr(5, 9), (2, -2): Fr(10, 27), (2, -1): Fr(2, 9), (2, 0): Fr(1, 9), (2, 1): Fr(1, 27),
        (3, -3): Fr(1, 9), (3, -2): Fr(5, 27), (3, -1): Fr(2, 9), (3, 0): Fr(2, 9), (3, 1): Fr(5, 27), (3, 2): Fr(1, 9),
    },
    -1: {  # Tabla 17, sigma- (m_F' = m_F - 1)
        (2, -1): Fr(1, 27), (2, 0): Fr(1, 9), (2, 1): Fr(2, 9), (2, 2): Fr(10, 27), (2, 3): Fr(5, 9),
        (3, -2): Fr(1, 9), (3, -1): Fr(5, 27), (3, 0): Fr(2, 9), (3, 1): Fr(2, 9), (3, 2): Fr(5, 27), (3, 3): Fr(1, 9),
    },
}

print(" F'  m_F  m_F'   peso (x27)   corrimiento (MHz/G)")
for Fp, m, mp in saltos(+1):
    p = peso[+1][(Fp, m)]
    print(f" {Fp}   {m:+d}   {mp:+d}      {int(p * 27):3d}          {corrimiento_salto(Fp, m, mp):+.3f}")

 F'  m_F  m_F'   peso (x27)   corrimiento (MHz/G)
 2   -3   -2       15          +1.711
 2   -2   -1       10          +1.089
 2   -1   +0        6          +0.467
 2   +0   +1        3          -0.156
 2   +1   +2        1          -0.778
 3   -3   -2        3          +1.089
 3   -2   -1        5          +0.778
 3   -1   +0        6          +0.467
 3   +0   +1        6          +0.156
 3   +1   +2        5          -0.156
 3   +2   +3        3          -0.467


Hacia $F' = 2$ los pesos suman $35/27$, y hacia $F' = 3$, $28/27$. Su cociente, $5/4$, es el de los factores $S_{32} = 5/9$ y $S_{33} = 4/9$ de la Tabla 8: las Tablas 15 y 17 ya incluyen cómo se reparte la absorción entre los dos $F'$.

Los saltos más fuertes ($m_F = -3$ y $-2$ hacia $F' = 2$) son también los que más se corren hacia frecuencias mayores. Por eso, con $\sigma^+$, el pozo entero se corre hacia arriba.

## 7. Cuánto se corre el pozo

Cada salto, por sí solo, produce un pozo ancho: por el efecto Doppler, mide unos 475 MHz. Los pozos de los 11 saltos están corridos entre sí apenas unas decenas de MHz, así que se superponen y forman un solo pozo.

Ese pozo queda en el promedio de los corrimientos de los saltos, y los que más absorben pesan más en el promedio:

$$\frac{\Delta\nu_{\sigma^\pm}}{B} = \frac{\sum_i p_i\, c_i}{\sum_i p_i},$$

donde $p_i$ es el peso de cada salto y $c_i = \Delta\nu/B$, su corrimiento.

In [6]:
def corrimiento_pozo(q):
    suma_pc = sum(float(peso[q][(Fp, m)]) * corrimiento_salto(Fp, m, mp) for Fp, m, mp in saltos(q))
    suma_p = sum(float(peso[q][(Fp, m)]) for Fp, m, mp in saltos(q))
    return suma_pc / suma_p

c_mas = corrimiento_pozo(+1)
c_menos = corrimiento_pozo(-1)
separacion = c_mas - c_menos
print(f"pozo sigma+ : {c_mas:+.3f} MHz/G")
print(f"pozo sigma- : {c_menos:+.3f} MHz/G")
print(f"separación  : {separacion:.3f} MHz/G")

pozo sigma+ : +0.743 MHz/G
pozo sigma- : -0.743 MHz/G
separación  : 1.486 MHz/G


Con $\sigma^-$ los pesos son los mismos en orden inverso y todos los corrimientos cambian de signo, así que su pozo se corre lo mismo hacia el otro lado. Los pozos de los dos canales se separan, entonces, 1,49 MHz por gauss.

## 8. El campo

Dividimos por este valor las dos separaciones medidas. Al invertir los imanes, el campo cambia de signo pero no de tamaño. Por eso tomamos la mitad de la diferencia entre los dos campos, lo que además cancela cualquier separación que no dependa del campo.

El error que da el notebook es solo el estadístico, que propagamos de las separaciones. El informe le suma lo que cambia el resultado al variar el ancho y la ventana del ajuste.

In [7]:
from math import hypot

delta_con, err_con = -122.3, 12.0          # MHz, imanes en un sentido
delta_inv, err_inv = +112.2, 10.0          # MHz, imanes invertidos

B_con = delta_con / separacion
B_inv = delta_inv / separacion
B = (B_inv - B_con) / 2
err_B = hypot(err_con, err_inv) / separacion / 2

print(f"B con imanes      = {B_con:+.0f} ± {err_con / separacion:.0f} G")
print(f"B invertidos      = {B_inv:+.0f} ± {err_inv / separacion:.0f} G")
print(f"|B|               = {B:.0f} ± {err_B:.0f} G (solo estadístico)")

B con imanes      = -82 ± 8 G
B invertidos      = +76 ± 7 G
|B|               = 79 ± 5 G (solo estadístico)
